## Metadata Loading and Source Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

path = "/kaggle/input/datasets/anastasiapetrunia/ukrainian-handwriting-ocr-dataset/"
TRAIN_META_PATH = path+"train/metadata.jsonl"
SILVER_META_PATH = path+"silver/metadata.jsonl"

def eda(meta_path):
    # ==========================================
    # STEP 1: Load & Inspect Metadata
    # ==========================================
    print("Loading metadata...")
    df = pd.read_json(meta_path, lines=True)
    
    # Calculate total regions by checking the length of the list in the 'regions' column
    df['num_regions'] = df['regions'].apply(lambda x: len(x) if isinstance(x, list) else 0)
    
    print(f"Total Images: {len(df)}")
    print(f"Total Regions (Bounding Boxes): {df['num_regions'].sum()}")
    print("-" * 50)
    
    # ==========================================
    # STEP 2: Analyze the 4 Sources
    # ==========================================
    # Calculate Aspect Ratio to understand capture methods (Phone vs Scanner)
    df['aspect_ratio'] = df['image_width'] / df['image_height']
    df['resolution_megapixels'] = (df['image_width'] * df['image_height']) / 1_000_000
    
    # Group by source to see the distribution of the dataset
    source_analysis = df.groupby('source').agg(
        image_count=('file_name', 'count'),
        total_regions=('num_regions', 'sum'),
        avg_regions_per_image=('num_regions', 'mean'),
        avg_width=('image_width', 'mean'),
        avg_height=('image_height', 'mean'),
        avg_megapixels=('resolution_megapixels', 'mean'),
        avg_aspect_ratio=('aspect_ratio', 'mean')
    ).round(2).reset_index()
    
    # Sort by image count for readability
    source_analysis = source_analysis.sort_values('image_count', ascending=False)
    
    print("\n--- Source Level Breakdown ---")
    display(source_analysis)
    
    # --- Visualization for Quick Insights ---
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    sns.set_theme(style="whitegrid")
    
    # 1. Image Distribution
    sns.barplot(data=source_analysis, x='source', y='image_count', ax=axes[0], palette="viridis", hue='source')
    axes[0].set_title("Number of Images per Source")
    axes[0].set_ylabel("Count")
    
    # 2. Region Density
    sns.barplot(data=source_analysis, x='source', y='avg_regions_per_image', ax=axes[1], palette="magma", hue='source')
    axes[1].set_title("Avg Regions (BBoxes) per Image")
    axes[1].set_ylabel("Regions")
    
    # 3. Resolution (Megapixels)
    sns.boxplot(data=df, x='source', y='resolution_megapixels', ax=axes[2], palette="crest", hue='source')
    axes[2].set_title("Image Resolution Distribution (MP)")
    axes[2].set_ylabel("Megapixels")
    
    plt.tight_layout()
    plt.show()

    return df

### Examining train data

In [ ]:
train_df = eda(TRAIN_META_PATH)

### Examining silver data

In [ ]:
silver_df = eda(SILVER_META_PATH)

## Important Findings from Metadata

1. The "Dictation" Domain Gap (Silver Anomaly)
   - Finding: The silver dataset contains zero dictation images, while train is dominated by them.
   - The Risk: Pre-training a model purely on silver will cause a "cold start" failure when it encounters National Dictation photos in the test set.
   - Actionable: The organizers expect us to use pseudo-labeling. We must use the publicly available canonical texts for these dictations to align text-lines and generate our own silver data.
2. The Aspect Ratio Trap
   - Finding: Aspect ratios vary wildly. university exams are tall and narrow (0.71), while school homework images are almost perfectly square (0.97 - 0.99).
   - The Risk: Blindly resizing images to a standard square (e.g., 640 x 640 or 1024 x 1024) for a YOLO detector will heavily compress and squish the text horizontally, destroying the handwriting geometry.
   - Actionable: Letterbox resizing (padding with gray/black bars to maintain the original aspect ratio) is strictly mandatory for this pipeline.
3. Silver Bounding Box Reliability
   - Finding: The average number of regions per image is nearly identical between the human-annotated train set and the auto-annotated silver set (e.g., Archive: 21.8 vs 22.1).
   - Actionable: Stage 1 of the auto-annotation pipeline (Qwen3-VL) did an excellent job. We can safely trust the silver bounding boxes to pre-train the object detection (region cropping) phase of our model.
4. Massive Distribution Imbalance
   - Finding: The silver dataset is heavily skewed toward historical documents (60% archive). Conversely, archive is the smallest category in the train set.
   - The Risk: If you train on the raw merged dataset, your model will heavily overfit to 1920s faded ink and archaic orthography.
   - Actionable: Implement strict Stratified/Weighted Sampling in your PyTorch DataLoader to ensure the model sees an equal balance of modern phone photos and historical scans during fine-tuning.
5. Resolution Disparity
   - Finding: Phone-captured school images average ~12 Megapixels, while scanned archive documents average only ~5-6 Megapixels.
   - Actionable: Applying aggressive downsampling or blurring augmentations might be fine for the 12MP school data, but it risks completely erasing the already faint, thin pen strokes in the lower-resolution archival data. Augmentations must be source-aware.

In [ ]:
import os
from PIL import Image
import matplotlib.patches as patches

# ==========================================
# STEP 3: Region Type & Legibility Breakdown
# ==========================================
def region_analysis(image_dir, df):
    print("Exploding regions for deep analysis...")
    # We explode the list of dictionaries into separate rows to analyze individual regions
    df_regions = df.explode('regions').dropna(subset=['regions'])
    
    # Extract the keys from the dictionary into their own columns
    df_regions['type'] = df_regions['regions'].apply(lambda x: x.get('type'))
    df_regions['text'] = df_regions['regions'].apply(lambda x: x.get('text'))
    df_regions['language'] = df_regions['regions'].apply(lambda x: x.get('language'))
    df_regions['legibility'] = df_regions['regions'].apply(lambda x: x.get('legibility'))
    
    print("\n--- Region Types by Source ---")
    type_breakdown = pd.crosstab(df_regions['source'], df_regions['type'])
    display(type_breakdown)
    
    print("\n--- Scorable Regions for CER (uk + legible + text) ---")
    # Filter for scorable regions: uk language, legible, and not an image/graph
    scorable = df_regions[
        (df_regions['language'] == 'uk') & 
        (df_regions['legibility'] == 'legible') & 
        (~df_regions['type'].isin(['image', 'graph']))
    ]
    scorable_counts = scorable.groupby('source').size()
    total_counts = df_regions.groupby('source').size()
    scorable_pct = (scorable_counts / total_counts * 100).round(1)
    
    scorable_df = pd.DataFrame({
        'Total Regions': total_counts,
        'Scorable Regions': scorable_counts,
        '% Scorable': scorable_pct
    })
    display(scorable_df)
    
    # ==========================================
    # STEP 4: Visualize Samples with BBoxes
    # ==========================================
    
    def plot_random_sample(source_name, df_meta):
        """Plots a random image from a specific source with its bounding boxes."""
        sample = df_meta[df_meta['source'] == source_name].sample(1).iloc[0]
        img_path = os.path.join(image_dir, sample['file_name'])
        
        if not os.path.exists(img_path):
            print(f"Image not found: {img_path}. (Check your IMAGE_DIR path)")
            return
    
        img = Image.open(img_path)
        fig, ax = plt.subplots(figsize=(10, 10))
        ax.imshow(img)
        
        # Colors for different types
        color_map = {
            'handwritten': 'green', 'printed': 'blue', 'formula': 'red',
            'table': 'purple', 'annotation': 'orange', 'image': 'yellow', 'graph': 'cyan'
        }
    
        for region in sample['regions']:
            bbox = region['bbox'] # [x1, y1, x2, y2]
            r_type = region['type']
            x, y = bbox[0], bbox[1]
            width, height = bbox[2] - bbox[0], bbox[3] - bbox[1]
            
            color = color_map.get(r_type, 'white')
            rect = patches.Rectangle((x, y), width, height, linewidth=2, edgecolor=color, facecolor='none')
            ax.add_patch(rect)
            ax.text(x, y - 10, r_type, color=color, fontsize=10, backgroundcolor='black')
    
        plt.title(f"Source: {source_name} | {sample['file_name']}")
        plt.axis('off')
        plt.show()
    
    # Plot one example from each source
    print("\n--- Generating Visualizations ---")
    for src in df['source'].unique():
        plot_random_sample(src, df)

    return df_regions

## Exploring regions

### Train regions

In [ ]:
train_df_regions = region_analysis(path+"train", train_df)

### Silver regions

In [ ]:
silver_df_regions = region_analysis(path+"silver", silver_df)

In [ ]:
import re
import collections

# ==========================================
# STEP 5: Text Statistics & Character Set
# ==========================================
def text_stats(df_regions):
    print("Analyzing Text Statistics and Character Sets...")
    
    # Filter out regions without text (images, graphs, illegible)
    text_regions = df_regions[df_regions['text'].notna() & (df_regions['text'] != "")]
    
    # 1. Text Length Analysis (Important for sequence modeling)
    text_regions = text_regions.copy()
    text_regions['text_length'] = text_regions['text'].apply(len)
    
    length_stats = text_regions.groupby('source')['text_length'].agg(['mean', 'max', 'min']).round(1)
    print("\n--- Text Length (Characters per Region) by Source ---")
    display(length_stats)
    
    # 2. Global Character Frequency
    all_text = "".join(text_regions['text'].tolist())
    char_counts = collections.Counter(all_text)
    
    # Separate into categories for easier reading
    cyrillic_chars = {k: v for k, v in char_counts.items() if re.match(r'[А-Яа-яЄєІіЇїҐґ]', k)}
    latin_chars = {k: v for k, v in char_counts.items() if re.match(r'[A-Za-z]', k)}
    digits = {k: v for k, v in char_counts.items() if k.isdigit()}
    # Everything else (punctuation, math symbols, spaces)
    other_chars = {k: v for k, v in char_counts.items() if k not in cyrillic_chars and k not in latin_chars and k not in digits}
    
    print(f"\nTotal Unique Characters (Vocabulary Size): {len(char_counts)}")
    print(f"Cyrillic Letters: {len(cyrillic_chars)}")
    print(f"Latin Letters: {len(latin_chars)}")
    print(f"Digits: {len(digits)}")
    print(f"Symbols/Punctuation: {len(other_chars)}")
    
    # 3. The "Archive" vs Modern Difference (Finding archaic letters)
    archive_text = "".join(text_regions[text_regions['source'] == 'archive']['text'].tolist())
    modern_text = "".join(text_regions[text_regions['source'] != 'archive']['text'].tolist())
    
    archive_chars = set(archive_text)
    modern_chars = set(modern_text)
    
    archaic_only = archive_chars - modern_chars
    print(f"\n--- Characters found ONLY in Archive data ---")
    print(sorted(list(archaic_only)))
    
    # 4. Special Markers (Strikethrough)
    strikethrough_count = text_regions['text'].str.contains('~~', regex=False).sum()
    print(f"\n--- Special Markers ---")
    print(f"Regions containing ~~ (strikethrough): {strikethrough_count}")
    
    # 5. Print Top 20 Rarest Cyrillic Characters (Potential weaknesses)
    sorted_cyrillic = sorted(cyrillic_chars.items(), key=lambda x: x[1])
    print("\n--- Top 20 RAREST Cyrillic Characters ---")
    for char, count in sorted_cyrillic[:20]:
        print(f"'{char}': {count} occurrences")

## Exploring text stats

### Train text stats

In [ ]:
text_stats(train_df_regions)

### Silver text stats

In [ ]:
text_stats(silver_df_regions)

## Analyzing text outliers

In [ ]:
def analyze_outliers(df_regions, df_meta, image_dir, length_threshold=500, num_samples=3):
    """
    Finds and plots images containing text regions larger than the threshold.
    Highlights outlier boxes in red and prints the hallucinated text below.
    """
    print(f"Searching for outlier regions (>{length_threshold} characters)...")
    
    # 1. Explode to find the specific images containing outliers
    # df_regions = df_silver_meta.explode('regions').dropna(subset=['regions'])
    # df_regions['text'] = df_regions['regions'].apply(lambda x: x.get('text', '') if x.get('text') else '')
    df_regions['text_length'] = df_regions['text'].apply(len)
    
    # Get unique filenames of images that contain at least one outlier
    outlier_files = df_regions[df_regions['text_length'] > length_threshold]['file_name'].unique()
    
    print(f"⚠️ Found {len(outlier_files)} images containing massive text blocks.")
    if len(outlier_files) == 0:
        return

    # 2. Plot the requested number of samples
    for file_name in outlier_files[:num_samples]:
        img_path = os.path.join(image_dir, file_name)
        if not os.path.exists(img_path):
            print(f"Image not found at path: {img_path}")
            continue
            
        img = Image.open(img_path)
        fig, ax = plt.subplots(figsize=(12, 12))
        ax.imshow(img)
        
        # Get the original region list for this image
        image_data = df_meta[df_meta['file_name'] == file_name].iloc[0]
        
        print(f"\n{'='*60}")
        print(f"🖼️ Analzying Image: {file_name} (Source: {image_data['source']})")
        print(f"{'='*60}")
        
        for i, region in enumerate(image_data['regions']):
            bbox = region['bbox']
            text = region.get('text', '')
            r_type = region.get('type', 'unknown')
            
            x, y = bbox[0], bbox[1]
            width, height = bbox[2] - bbox[0], bbox[3] - bbox[1]
            
            # 3. Highlight Logic: Red for Outliers, Green for Normal
            is_outlier = text and len(text) > length_threshold
            color = 'red' if is_outlier else 'limegreen'
            linewidth = 3 if is_outlier else 1
            
            # Draw the box
            rect = patches.Rectangle((x, y), width, height, linewidth=linewidth, edgecolor=color, facecolor='none')
            ax.add_patch(rect)
            
            # Add a small label tag (e.g., "Box 0") so we can map it to the text below
            ax.text(x, y - 10, f"Box {i}", color='white', fontsize=10, backgroundcolor=color, fontweight='bold')
            
            # 4. Print the text below the plot (truncating if it's monstrous)
            if is_outlier:
                print(f"\n🚨 OUTLIER [Box {i}] | Type: {r_type} | Length: {len(text)} chars")
                unique_chars = sorted(list(set(text)))
                num_unique = len(unique_chars)
                chars_display = "".join(unique_chars)
                
                print(f"🔤 Unique Characters ({num_unique}): {chars_display}")
                print(f"-"*40)
                
                if len(text) > 500:
                    preview = text[:250] + "\n\n... [TRUNCATED] ...\n\n" + text[-250:]
                else:
                    preview = text
                print(f"Text:\n{preview}")
            elif text:
                # Optional: Uncomment if you want to see the "normal" text too
                # print(f"✅ Normal [Box {i}] | Length: {len(text)} chars | Text: {text[:50]}...")
                pass

        plt.title(f"Silver Error Analysis: {file_name}\nRed = Outlier Box (> {length_threshold} chars)", color='red', pad=15)
        plt.axis('off')
        plt.show()

### Silver outliers

In [ ]:
analyze_outliers(silver_df_regions, silver_df, path+"silver", length_threshold=1000, num_samples=3)

### Train outliers

In [ ]:
analyze_outliers(train_df_regions, train_df, path+"train", length_threshold=400, num_samples=3)

### Text Length Percentiles

In [ ]:
def analyze_text_length(df_regions):
    # Calculate the 90th, 95th, and 99th percentiles for text length
    # Drop extreme outliers in silver dataset for the plot
    text_regions = df_regions[df_regions['text'].notna() & (df_regions['text'] != "")& (df_regions['text_length'] < 1000)]
    percentiles = text_regions.groupby('source')['text_length'].quantile([0.90, 0.95, 0.99]).unstack()
    percentiles.columns = ['90th Percentile', '95th Percentile', '99th Percentile']
    
    print("--- Exact Text Length Percentiles (Train Set) ---")
    display(percentiles.round(1))

    max_length = percentiles["99th Percentile"].max()
    plt.figure(figsize=(10, 4))
    sns.histplot(data=text_regions, x='text_length', hue='source', bins=100, kde=True)
    plt.xlim(0, max_length)
    plt.title("Distribution of Text Lengths (Train Set)")
    plt.xlabel("Number of Characters")
    plt.ylabel("Frequency")
    plt.show()

#### Train text lenght analysis

In [ ]:
analyze_text_length(train_df_regions)

#### Silver text lenght analysis

In [ ]:
analyze_text_length(silver_df_regions)

## Important Findings from Region Analysis & Text Stats

1. The Silver "Token Loop" Hallucination

    - Finding: The massive 48,864-character text blocks in the silver dataset aren't actual paragraphs; they are infinite loops of repeating punctuation (like dots . and hyphens -).
    
    - The Cause: This is a classic vulnerability in Large Vision-Language Models (like Gemini/Qwen). When faced with visual repeating patterns (like a dotted fill-in-the-blank line or a page crease), the model's auto-regressive decoding gets stuck in a "repetition loop," predicting the same token until it hits a hard context limit.
    
    - Actionable: We must implement a Repetition Filter. Any region containing 10+ consecutive identical punctuation marks must be discarded from the silver set before training.

2. Data-Driven Thresholding (The 99th Percentile)

    - Finding: By calculating the exact distributions of the train data, we established that the absolute longest 99th percentile for text length is 255 characters for silver data and 160.2 for train data.
    
    - The Risk: Setting an arbitrary sequence length limit (like 128) based on a guess would truncate valid mathematical formulas and complex archive lines, directly hurting the Page CER score.
    
    - Actionable: The scientifically safe sequence length threshold for this dataset is 260. We must apply this hard limit in our silver cleaning script to safely drop the VLM text hallucinations while preserving 99.9% of all valid human handwriting.

3. The Train "Granularity Drift"

    - Finding: Image 00411.jpg in the human-annotated train set contains a legitimate 525-character block of text inside a single bounding box, instead of being broken down line-by-line.
    
    - The Cause: Human annotator inconsistency. For 99% of the dataset, 1 box = 1 line. For a few rare documents, the annotator drew one box for a whole paragraph.
    
    - Actionable (Metric Advantage): The Page CER metric (50% of the score) is agnostic to bounding box granularity—it just sorts the boxes and concatenates the text.
    
    - Strategy: Because we scientifically set our model's max_length to 200 based on the 99th percentile, we simply accept that the model will truncate the 00411.jpg outlier during training. We sacrifice a single badly-annotated image to avoid blowing up our GPU memory to accommodate 500+ character sequences

In [ ]:
def create_ocr_training_datasets(df_train, df_silver, max_seq_length=260):
    """
    Filters train and silver datasets to create clean, safe subsets 
    specifically for training the text transcription (OCR) model.
    """
    print("Extracting valid training vocabulary...")
    # 1. Build the canonical vocabulary from the Train dataset
    # We only look at regions that actually have text
    train_regions = df_train.explode('regions').dropna(subset=['regions'])
    train_texts = [r.get('text', '') for r in train_regions['regions'] if r.get('text')]
    valid_train_vocab = set("".join(train_texts))
    
    def is_valid_train_region(region):
        text = region.get('text', '')
        # Drop empty text (images, graphs, illegible, language=other)
        if not text: 
            return False
        # Drop the massive 500+ character outliers to save GPU memory
        if len(text) > max_seq_length:
            return False
        return True

    def is_valid_silver_region(region):
        text = region.get('text', '')
        r_type = region.get('type', '')
        
        # Rule 1: Must have text and fit in our max sequence length
        if not text or len(text) > max_seq_length:
            return False
            
        # Rule 2: The Repetition Filter (Kills the '.' and '-' VLM token loops)
        if re.search(r'(.)\1{9,}', text):
            return False
            
        # Rule 3: The Foreign Language / Hallucination Filter
        # If the region contains more than 2 characters outside the known train alphabet, drop it
        out_of_vocab_chars = [c for c in text if c not in valid_train_vocab]
        if len(out_of_vocab_chars) > 2:
            return False
            
        return True

    def filter_dataframe(df, filter_func, name):
        kept_records = []
        total_regions = 0
        kept_regions = 0
        
        for _, row in df.iterrows():
            total_regions += len(row['regions'])
            
            # Apply the specific filter function to the regions list
            valid_regions = [r for r in row['regions'] if filter_func(r)]
            kept_regions += len(valid_regions)
            
            # Keep the image metadata only if it still has valid text regions to train on
            if len(valid_regions) > 0:
                clean_row = row.copy()
                clean_row['regions'] = valid_regions
                kept_records.append(clean_row)
                
        df_clean = pd.DataFrame(kept_records)
        
        print(f"\n--- {name} Filtering Results ---")
        print(f"Original Images: {len(df)} -> Clean Images: {len(df_clean)}")
        print(f"Total Regions:   {total_regions}")
        print(f"Regions Kept:    {kept_regions} ({(kept_regions/total_regions*100):.1f}%)")
        print(f"Regions Dropped: {total_regions - kept_regions}")
        
        return df_clean

    # 2. Execute the filters
    print("\nFiltering Train Data (Gold)...")
    clean_train = filter_dataframe(df_train, is_valid_train_region, "Train")
    
    print("\nFiltering Silver Data (Pseudo-labels)...")
    clean_silver = filter_dataframe(df_silver, is_valid_silver_region, "Silver")
    
    return clean_train, clean_silver

In [ ]:
df_train_clean, df_silver_clean = create_ocr_training_datasets(train_df, silver_df, max_seq_length=260)

# Optional: Save to disk for your training notebook
df_train_clean.to_json('/kaggle/working/train_clean_metadata.jsonl', orient='records', lines=True)
df_silver_clean.to_json('/kaggle/working/silver_clean_metadata.jsonl', orient='records', lines=True)